# Exp15b — Multi-language agreement test (Gemma-2 2B/9B/27B)

**Question.** Does a high-epsilon task (subject-verb agreement) dissociate across languages with scale, while IOI (Exp14) stays shared?

**Languages.** FR, DE (close to EN); HI, AR, SW (distant). Yoruba and Chinese are excluded: no verb agreement, so the task does not exist there.

**Pipeline per model, per language.** Held-out head ranking (batched) -> ablation table -> EN->lang head-output patching with a same-language different-template floor -> compat = (patched - ablated)/(clean - ablated).

**Fixes vs Exp15:** batched ranking, ranking on held-out prompts, verbs filtered *before* prompt generation (so EN/target prompts stay index-aligned), self-patch identity test, 5 languages.

**Read before trusting anything:** (1) the self-test cell must pass; (2) baseline acc >= 0.75 for a (model, language) cell or it is skipped; (3) floor compat ~1.0 or the cross-language number at that cell is not interpretable. Templates and nouns for HI/AR/SW/DE were written without native-speaker review. Have someone check them, or treat the baseline gate as the first filter.


## Cell 1 - Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json, random, gc
from pathlib import Path
from collections import defaultdict
from contextlib import contextmanager

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)

MODEL_PATHS = {
    "2b_base": "google/gemma-2-2b",
    "9b_base": "/home/models/gemma-2-9b",
    "9b_it":   "/home/models/gemma-2-9b-it",
    "27b_it":  "/home/models/gemma-2-27b-it",
}

TARGET_LANGS = ["fr", "de", "hi", "ar", "sw"]
SEEDS = [0, 7, 13]          # eval prompt sets
RANK_SEED, RANK_N = 500, 40 # held-out ranking prompts (disjoint seed)
N_PROMPTS = 120
TOPK = 10
GATE_ACC = 0.75
RANK_TARGET_LANGS = True    # also rank heads per target language (extra cost, gives lang-heads -> lang ablation)

OUT_DIR = Path("exp15b_results"); OUT_DIR.mkdir(exist_ok=True)
EXP14_DIR = Path("exp14_results")
print(DEVICE, torch.cuda.is_available())


/home/anwoy/miniconda3/envs/ke/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda True


## Cell 2 - Language registry

Structure `[Det] N1(sg) PREP [Det] N2(pl)` then the verb. Correct next token = singular form (agrees with N1); plural form = attractor-driven error. `pool="human"` restricts the EN-noun *matched* condition to human nouns for languages where the verb needs an animate subject (AR, SW). All verb lists have 3 pairs so the verb index aligns across languages.

In [2]:
EN_PAIRS = [
    ("cat","cats"),("dog","dogs"),("book","books"),("key","keys"),
    ("hat","hats"),("bird","birds"),("horse","horses"),("phone","phones"),
    ("doctor","doctors"),("teacher","teachers"),("student","students"),("friend","friends"),
    ("brother","brothers"),("boy","boys"),("father","fathers"),("player","players"),
    ("author","authors"),("painter","painters"),("singer","singers"),("writer","writers"),
]
EN_POOL = {"all": EN_PAIRS, "human": EN_PAIRS[8:]}

LANGS = {
 "en": dict(
    templates=["The {N1} near the {N2}","The {N1} behind the {N2}","The {N1} next to the {N2}",
               "The {N1} by the {N2}","The {N1} with the {N2}","The {N1} among the {N2}"],
    pairs=EN_PAIRS,
    verbs=[(" is"," are"),(" was"," were"),(" has"," have")], pool="all"),
 "fr": dict(
    templates=["Le {N1} près des {N2}","Le {N1} derrière les {N2}","Le {N1} à côté des {N2}",
               "Le {N1} par les {N2}","Le {N1} avec les {N2}","Le {N1} parmi les {N2}"],
    pairs=[("chat","chats"),("chien","chiens"),("livre","livres"),("stylo","stylos"),
           ("chapeau","chapeaux"),("oiseau","oiseaux"),("cheval","chevaux"),("téléphone","téléphones"),
           ("docteur","docteurs"),("professeur","professeurs"),("étudiant","étudiants"),("ami","amis"),
           ("frère","frères"),("garçon","garçons"),("père","pères"),("joueur","joueurs"),
           ("auteur","auteurs"),("peintre","peintres"),("chanteur","chanteurs"),("écrivain","écrivains")],
    verbs=[(" est"," sont"),(" était"," étaient"),(" a"," ont")], pool="all"),
 "de": dict(  # plural nouns in dative (after neben/hinter/bei/unter/mit/zwischen)
    templates=["Der {N1} neben den {N2}","Der {N1} hinter den {N2}","Der {N1} bei den {N2}",
               "Der {N1} unter den {N2}","Der {N1} mit den {N2}","Der {N1} zwischen den {N2}"],
    pairs=[("Hund","Hunden"),("Vogel","Vögeln"),("Lehrer","Lehrern"),("Arzt","Ärzten"),
           ("Freund","Freunden"),("Bruder","Brüdern"),("Junge","Jungen"),("Vater","Vätern"),
           ("Spieler","Spielern"),("Autor","Autoren"),("Maler","Malern"),("Sänger","Sängern"),
           ("Schüler","Schülern"),("Mann","Männern"),("Nachbar","Nachbarn"),("Student","Studenten"),
           ("Fahrer","Fahrern"),("Koch","Köchen"),("Bauer","Bauern"),("Schreiber","Schreibern")],
    verbs=[(" ist"," sind"),(" war"," waren"),(" hat"," haben")], pool="all"),
 "hi": dict(  # SOV; masculine nouns; plural N2 in oblique; present/past/habitual copula
    templates=["{N1} {N2} के पास","{N1} {N2} के पीछे","{N1} {N2} के साथ",
               "{N1} {N2} के बीच","{N1} {N2} के सामने","{N1} {N2} के ऊपर"],
    pairs=[("कुत्ता","कुत्तों"),("लड़का","लड़कों"),("घोड़ा","घोड़ों"),("बच्चा","बच्चों"),
           ("डॉक्टर","डॉक्टरों"),("शिक्षक","शिक्षकों"),("छात्र","छात्रों"),("दोस्त","दोस्तों"),
           ("भाई","भाइयों"),("पक्षी","पक्षियों"),("खिलाड़ी","खिलाड़ियों"),("लेखक","लेखकों"),
           ("गायक","गायकों"),("चित्रकार","चित्रकारों"),("पड़ोसी","पड़ोसियों"),("किसान","किसानों"),
           ("आदमी","आदमियों"),("कमरा","कमरों"),("पेड़","पेड़ों"),("मज़दूर","मज़दूरों")],
    verbs=[(" है"," हैं"),(" था"," थे"),(" होता"," होते")], pool="all"),
 "ar": dict(  # SVO nominal subject + imperfect verb; human nouns; plural N2 genitive
    templates=["{N1} بجانب {N2}","{N1} خلف {N2}","{N1} مع {N2}",
               "{N1} بين {N2}","{N1} قرب {N2}","{N1} أمام {N2}"],
    pairs=[("الطبيب","الأطباء"),("المعلم","المعلمين"),("الطالب","الطلاب"),("الصديق","الأصدقاء"),
           ("الولد","الأولاد"),("اللاعب","اللاعبين"),("الكاتب","الكتّاب"),("الرسام","الرسامين"),
           ("المغني","المغنين"),("الجار","الجيران"),("الرجل","الرجال"),("المهندس","المهندسين"),
           ("الفلاح","الفلاحين"),("الطفل","الأطفال"),("الأب","الآباء"),("الأخ","الإخوة"),
           ("العامل","العمال"),("الجندي","الجنود"),("السائق","السائقين"),("الممثل","الممثلين")],
    verbs=[(" يعمل"," يعملون"),(" يلعب"," يلعبون"),(" يكتب"," يكتبون")], pool="human"),
 "sw": dict(  # class 1/2 (m-/wa-) human nouns; subject prefix a-/wa-; no determiners
    templates=["{N1} karibu na {N2}","{N1} nyuma ya {N2}","{N1} pamoja na {N2}",
               "{N1} kando ya {N2}","{N1} mbele ya {N2}","{N1} kati ya {N2}"],
    pairs=[("mwalimu","walimu"),("mwanafunzi","wanafunzi"),("mtoto","watoto"),("mtu","watu"),
           ("mkulima","wakulima"),("mwandishi","waandishi"),("mchezaji","wachezaji"),("mwimbaji","waimbaji"),
           ("msanii","wasanii"),("mpishi","wapishi"),("mvuvi","wavuvi"),("mzee","wazee"),
           ("mgeni","wageni"),("mwanajeshi","wanajeshi"),("mfanyakazi","wafanyakazi"),("msafiri","wasafiri"),
           ("mchungaji","wachungaji"),("mwanasayansi","wanasayansi"),("msichana","wasichana"),("mvulana","wavulana")],
    verbs=[(" anasoma"," wanasoma"),(" anacheza"," wanacheza"),(" anaandika"," wanaandika")], pool="human"),
}
assert all(len(v["verbs"]) == 3 and len(v["templates"]) == 6 for v in LANGS.values())


## Cell 3 - Prompt builder (index-aligned across languages)

In [3]:
def gen_idx(n, seed, shift_tpl=False):
    rng = random.Random(seed)
    idx = [(rng.randrange(6), rng.randrange(10**6), rng.randrange(10**6), rng.randrange(10**6)) for _ in range(n)]
    if shift_tpl:
        idx = [((t + 1) % 6, a, b, v) for (t, a, b, v) in idx]
    return idx

def build_prompts(spec, noun_pairs, verbs, idx):
    out = []
    for (t, a, b, v) in idx:
        i1 = a % len(noun_pairs)
        i2 = (i1 + 1 + (b % (len(noun_pairs) - 1))) % len(noun_pairs)  # i2 != i1
        n1, n2 = noun_pairs[i1][0], noun_pairs[i2][1]
        vs, vp = verbs[v % len(verbs)]
        out.append({"text": spec["templates"][t].format(N1=n1, N2=n2),
                    "correct": vs, "incorrect": vp})
    return out

def make_sets(verbs, seed, n=N_PROMPTS, shift_tpl=False, langs=None):
    idx = gen_idx(n, seed, shift_tpl)
    langs = langs or TARGET_LANGS
    sets = {"en": build_prompts(LANGS["en"], LANGS["en"]["pairs"], verbs["en"], idx)}
    for lg in langs:
        if not verbs.get(lg):
            continue
        sp = LANGS[lg]
        sets[lg + "_nat"] = build_prompts(sp, sp["pairs"], verbs[lg], idx)
        sets[lg + "_m"]   = build_prompts(sp, EN_POOL[sp["pool"]], verbs[lg], idx)
    return sets

_demo_v = {k: v["verbs"] for k, v in LANGS.items()}
for k, ps in make_sets(_demo_v, 0, n=2).items():
    for p in ps:
        print(f"{k:7s} {p['text']!r}  correct={p['correct']!r} wrong={p['incorrect']!r}")


en      'The brother by the friends'  correct=' was' wrong=' were'
en      'The boy near the cats'  correct=' is' wrong=' are'
fr_nat  'Le frère par les amis'  correct=' était' wrong=' étaient'
fr_nat  'Le garçon près des chats'  correct=' est' wrong=' sont'
fr_m    'Le brother par les friends'  correct=' était' wrong=' étaient'
fr_m    'Le boy près des cats'  correct=' est' wrong=' sont'
de_nat  'Der Schüler unter den Sängern'  correct=' war' wrong=' waren'
de_nat  'Der Mann neben den Hunden'  correct=' ist' wrong=' sind'
de_m    'Der brother unter den friends'  correct=' war' wrong=' waren'
de_m    'Der boy neben den cats'  correct=' ist' wrong=' sind'
hi_nat  'गायक लेखकों के बीच'  correct=' था' wrong=' थे'
hi_nat  'चित्रकार कुत्तों के पास'  correct=' है' wrong=' हैं'
hi_m    'brother friends के बीच'  correct=' था' wrong=' थे'
hi_m    'boy cats के पास'  correct=' है' wrong=' हैं'
ar_nat  'الفلاح بين المهندسين'  correct=' يلعب' wrong=' يلعبون'
ar_nat  'الطفل بجانب الأطباء'  correct=' 

## Cell 4 - Model utils, verb check, batched logit-diff

In [4]:
def load_model(path):
    tok = AutoTokenizer.from_pretrained(path)
    mdl = AutoModelForCausalLM.from_pretrained(path, dtype=torch.bfloat16, device_map=DEVICE).eval()
    cfg = {"n_layers": mdl.config.num_hidden_layers,
           "n_heads": mdl.config.num_attention_heads,
           "head_dim": mdl.config.head_dim}
    return mdl, tok, cfg

def unload(mdl):
    del mdl; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def verify_verbs(tok, pairs):
    """Keep a pair if the FIRST tokens differ (logit-diff is read on the first token).
    Multi-token verbs (e.g. SW prefix-carried agreement) are allowed and reported."""
    good = []
    for a, b in pairs:
        ta = tok.encode(a, add_special_tokens=False); tb = tok.encode(b, add_special_tokens=False)
        if ta and tb and ta[0] != tb[0]:
            good.append((a, b))
            if len(ta) > 1 or len(tb) > 1:
                print(f"    note: {a!r}/{b!r} multi-token ({len(ta)}/{len(tb)}); scoring first token")
        else:
            print(f"    dropping {a!r}/{b!r}: first tokens {ta[:1]}/{tb[:1]}")
    return good

def cfg_of(mdl):
    return {"n_layers": mdl.config.num_hidden_layers, "n_heads": mdl.config.num_attention_heads,
            "head_dim": mdl.config.head_dim}

def first_ids(tok, prompts):
    c = torch.tensor([tok.encode(p["correct"], add_special_tokens=False)[0] for p in prompts], device=DEVICE)
    w = torch.tensor([tok.encode(p["incorrect"], add_special_tokens=False)[0] for p in prompts], device=DEVICE)
    return c, w


## Cell 5 - Head hooks

Pre-hook on `self_attn.o_proj`: input = concatenated head outputs `[B, S, n_heads*head_dim]`. Modes: `ablate` (zero heads, all positions), `capture` (store final-position output), `patch` (write payload at the final position; single-prompt only, no padding).

In [5]:
@contextmanager
def head_hooks(mdl, mode, heads, payload=None, store=None):
    cfg = cfg_of(mdl); hd = cfg["head_dim"]
    by_layer = defaultdict(list)
    for L, H in heads: by_layer[L].append(H)
    handles = []
    def mk(L, Hs):
        def pre(module, inputs):
            x = inputs[0]
            B, S, D = x.shape
            if mode == "capture":
                xv = x.view(B, S, D // hd, hd)
                for H in Hs: store[(L, H)] = xv[:, -1, H, :].detach().clone()
                return None
            xv = x.clone().view(B, S, D // hd, hd)
            if mode == "ablate":
                for H in Hs: xv[:, :, H, :] = 0.0
            elif mode == "patch":
                for H in Hs:
                    if (L, H) in payload:
                        xv[:, -1, H, :] = payload[(L, H)].to(xv.device, xv.dtype).reshape(-1, hd)
            return (xv.view(B, S, D),) + tuple(inputs[1:])
        return pre
    for L, Hs in by_layer.items():
        handles.append(mdl.model.layers[L].self_attn.o_proj.register_forward_pre_hook(mk(L, Hs)))
    try:
        yield
    finally:
        for h in handles: h.remove()

def bd(mdl, tok, prompts, heads=None, bs=40):
    """Batched logit-diff (correct - incorrect) at the final position; optional head ablation."""
    tok.padding_side = "left"
    c, w = first_ids(tok, prompts)
    out = []
    for s in range(0, len(prompts), bs):
        enc = tok([p["text"] for p in prompts[s:s+bs]], return_tensors="pt", padding=True).to(DEVICE)
        if heads:
            with head_hooks(mdl, "ablate", heads):
                lg = mdl(**enc).logits[:, -1].float()
        else:
            lg = mdl(**enc).logits[:, -1].float()
        ix = torch.arange(lg.shape[0], device=DEVICE)
        out.append((lg[ix, c[s:s+bs]] - lg[ix, w[s:s+bs]]).cpu().numpy())
    return np.concatenate(out)

def capture_payloads(mdl, tok, prompts, heads):
    tok.padding_side = "left"
    outs = []
    for p in prompts:
        enc = tok(p["text"], return_tensors="pt").to(DEVICE)
        st = {}
        with head_hooks(mdl, "capture", heads, store=st):
            mdl(**enc)
        outs.append({k: v.squeeze(0) for k, v in st.items()})
    return outs

def patched_diffs(mdl, tok, prompts, payloads):
    tok.padding_side = "left"
    res = np.zeros(len(prompts), dtype=np.float32)
    for i, (p, pl) in enumerate(zip(prompts, payloads)):
        enc = tok(p["text"], return_tensors="pt").to(DEVICE)
        with head_hooks(mdl, "patch", list(pl.keys()), payload=pl):
            lg = mdl(**enc).logits[0, -1].float()
        c = tok.encode(p["correct"], add_special_tokens=False)[0]
        w = tok.encode(p["incorrect"], add_special_tokens=False)[0]
        res[i] = float(lg[c] - lg[w])
    return res

def patch_compat(mdl, tok, donors, targets, heads):
    clean = bd(mdl, tok, targets).mean()
    abl = bd(mdl, tok, targets, heads=heads).mean()
    pay = capture_payloads(mdl, tok, donors, heads)
    pat = patched_diffs(mdl, tok, targets, pay).mean()
    gap = clean - abl
    comp = float((pat - abl) / gap) if abs(gap) > 1e-6 else float("nan")
    return {"compat": comp, "clean": float(clean), "ablated": float(abl), "patched": float(pat), "gap": float(gap)}


## Cell 6 - Self-test (run on 2B first, must pass)

Patching a prompt with *its own* captured head outputs must reproduce the clean logit-diff exactly (compat ~ 1). Also checks that ablating heads changes the output. If either fails, the hooks are broken and nothing downstream means anything.

In [6]:
def self_test(path="google/gemma-2-2b"):
    mdl, tok, cfg = load_model(path)
    v = {k: verify_verbs(tok, s["verbs"]) for k, s in LANGS.items()}
    ps = make_sets(v, 0, n=20, langs=["fr"])["en"]
    heads = [(cfg["n_layers"] // 2, 0), (cfg["n_layers"] // 2 + 1, 1), (cfg["n_layers"] // 2 + 2, 2)]
    clean = bd(mdl, tok, ps).mean()
    abl = bd(mdl, tok, ps, heads=heads).mean()
    own = patched_diffs(mdl, tok, ps, capture_payloads(mdl, tok, ps, heads)).mean()
    print(f"clean={clean:+.4f} ablated={abl:+.4f} self-patched={own:+.4f}")
    assert abs(own - clean) < 0.1 * max(1.0, abs(clean)), "self-patch != clean: hook/pairing bug"
    assert abs(abl - clean) > 1e-3, "ablation had no effect: hook not firing"
    print("self-test PASSED")
    unload(mdl)

# self_test()


## Cell 7 - Per-model pipeline

For each model: verify verbs per language -> baselines + gate -> held-out head ranking (EN, and optionally each target language) -> ablation table -> EN->lang patching with floor (EN->EN, donor template shifted by one). Averages over `SEEDS`. Results cached to `exp15b_results/<model>.json`; head rankings cached separately.

In [7]:
def rank_heads(mdl, tok, prompts, cfg):
    clean = bd(mdl, tok, prompts).mean()
    sc = np.zeros((cfg["n_layers"], cfg["n_heads"]), dtype=np.float32)
    for L in range(cfg["n_layers"]):
        for H in range(cfg["n_heads"]):
            sc[L, H] = clean - bd(mdl, tok, prompts, heads=[(L, H)]).mean()
    ranked = sorted([(L, H, float(sc[L, H])) for L in range(cfg["n_layers"]) for H in range(cfg["n_heads"])],
                    key=lambda x: -x[2])
    return {"clean": float(clean), "ranked": ranked}

def rank_or_load(mk, key, mdl, tok, prompts, cfg):
    f = OUT_DIR / f"{mk}_rank_{key}.json"
    if f.exists():
        return json.load(open(f))
    print(f"  ranking {mk}/{key}: {cfg['n_layers']*cfg['n_heads']} heads x batched passes")
    r = rank_heads(mdl, tok, prompts, cfg)
    json.dump(r, open(f, "w"))
    s = [x[2] for x in r["ranked"]]
    print(f"    top-5 scores {np.round(s[:5],2).tolist()} | median {np.median(s):.3f}")
    return r

def mean_std(xs):
    xs = [x for x in xs if x == x]
    return (float(np.mean(xs)), float(np.std(xs))) if xs else (float("nan"), float("nan"))

def run_model(mk, path, topk=TOPK):
    print(f"\n===== {mk} =====")
    mdl, tok, cfg = load_model(path)
    verbs = {lg: verify_verbs(tok, LANGS[lg]["verbs"]) for lg in ["en"] + TARGET_LANGS}
    langs = [lg for lg in TARGET_LANGS if verbs[lg]]
    assert verbs["en"], "no usable EN verbs"
    res = {"cfg": cfg, "langs": {}}

    rk = make_sets(verbs, RANK_SEED, RANK_N, langs=langs)
    en_rank = rank_or_load(mk, "en", mdl, tok, rk["en"], cfg)
    en_heads = [(L, H) for L, H, _ in en_rank["ranked"][:topk]]
    rng = random.Random(0)
    allh = [(L, H) for L in range(cfg["n_layers"]) for H in range(cfg["n_heads"])]
    rand_heads = rng.sample(allh, topk)

    for lg in langs:
        lg_heads = None
        if RANK_TARGET_LANGS:
            r = rank_or_load(mk, lg + "_m", mdl, tok, rk[lg + "_m"], cfg)
            lg_heads = [(L, H) for L, H, _ in r["ranked"][:topk]]
        acc_nat, acc_m, keep = [], [], True
        rows = defaultdict(list)
        for seed in SEEDS:
            S = make_sets(verbs, seed, langs=[lg]); Sa = make_sets(verbs, seed, shift_tpl=True, langs=[lg])
            en_p, tg = S["en"], S[lg + "_m"]
            d_en, d_tg, d_nat = bd(mdl, tok, en_p), bd(mdl, tok, tg), bd(mdl, tok, S[lg + "_nat"])
            acc_nat.append(float((d_nat > 0).mean())); acc_m.append(float((d_tg > 0).mean()))
            if acc_m[-1] < GATE_ACC:
                keep = False
                print(f"  [{lg}] gate FAIL seed {seed}: matched acc {acc_m[-1]:.3f} < {GATE_ACC}")
                continue
            ce, ct = d_en.mean(), d_tg.mean()
            ab = lambda hs, ps: bd(mdl, tok, ps, heads=hs).mean()
            rows["ret_en_en"].append(ab(en_heads, en_p) / ce)
            rows["ret_en_lang"].append(ab(en_heads, tg) / ct)
            rows["ret_rand_lang"].append(ab(rand_heads, tg) / ct)
            if lg_heads: rows["ret_lang_lang"].append(ab(lg_heads, tg) / ct)
            x = patch_compat(mdl, tok, en_p, tg, en_heads)
            f = patch_compat(mdl, tok, Sa["en"], en_p, en_heads)
            rows["compat"].append(x["compat"]); rows["gap"].append(x["gap"])
            rows["floor"].append(f["compat"])
        res["langs"][lg] = {"acc_nat": mean_std(acc_nat), "acc_matched": mean_std(acc_m), "gate_ok": keep,
                            **{k: mean_std(v) for k, v in rows.items()}}
        r_ = res["langs"][lg]
        print(f"  [{lg}] acc nat/matched {r_['acc_nat'][0]:.2f}/{r_['acc_matched'][0]:.2f} | "
              f"retained EN-heads->lang {r_.get('ret_en_lang',(float('nan'),))[0]:+.2f} "
              f"| compat {r_.get('compat',(float('nan'),))[0]:+.2f} | floor {r_.get('floor',(float('nan'),))[0]:+.2f}")
    json.dump(res, open(OUT_DIR / f"{mk}.json", "w"), indent=2)
    unload(mdl)
    return res

# Run one scale first (2B), inspect, then the rest:
# run_model("2b_base", MODEL_PATHS["2b_base"])
# for mk, p in MODEL_PATHS.items():
#     if mk != "2b_base": run_model(mk, p)


## Cell 8 - Summary (agreement vs IOI, by scale and language)

Reads `exp15b_results/*.json`. IOI compat is read from `exp14_results/<model>.json` if the layout is recognised (otherwise paste the Exp14 numbers into `IOI_COMPAT`).

Reading guide: only trust a cell where `gate_ok` and floor ~ 1.0. Under the epsilon prediction: IOI compat ~ 1 everywhere; agreement compat < 1, lower for HI/AR/SW than FR/DE, lower at larger scale.

In [8]:
IOI_COMPAT = {}   # e.g. {"2b_base": 1.14, "9b_base": 1.22, "9b_it": 1.16, "27b_it": 0.98}  (Exp14 EN->FR, fill if auto-load fails)

def load_ioi(mk):
    if mk in IOI_COMPAT: return IOI_COMPAT[mk]
    f = EXP14_DIR / f"{mk}.json"
    if not f.exists(): return None
    d = json.load(open(f))
    for path in [("compat","EN->FR"),("compat","en_to_fr"),("cross","compat"),("en_to_fr","compat")]:
        node = d
        try:
            for k in path: node = node[k]
            return float(node)
        except (KeyError, TypeError):
            continue
    print(f"  [{mk}] Exp14 JSON layout not recognised; set IOI_COMPAT manually.")
    return None

def summarize():
    hdr = f"{'model':8s} {'lang':4s} {'gate':5s} {'acc_m':>6s} {'ret EN->L':>10s} {'ret L->L':>9s} {'compat':>8s} {'floor':>7s} {'IOI':>7s}"
    print(hdr); print("-" * len(hdr))
    for mk in MODEL_PATHS:
        f = OUT_DIR / f"{mk}.json"
        if not f.exists(): continue
        r = json.load(open(f)); ioi = load_ioi(mk)
        for lg, x in r["langs"].items():
            g = lambda k: x[k][0] if k in x else float("nan")
            flag = "" if (x["gate_ok"] and abs(g("floor") - 1) < 0.15) else "  <- untrusted"
            print(f"{mk:8s} {lg:4s} {str(x['gate_ok']):5s} {x['acc_matched'][0]:6.2f} {g('ret_en_lang'):+10.2f} "
                  f"{g('ret_lang_lang'):+9.2f} {g('compat'):+8.2f} {g('floor'):+7.2f} {('%+.2f' % ioi) if ioi is not None else '   n/a':>7s}{flag}")

# summarize()


## Cell 9 - RUN (stage 1: self-test + 2B)

In [9]:
# ===== RUN =====  (execute cells 1-8 first, then this one)
# Stage 1: hook sanity test on 2B (fails loudly if hooks are broken)
self_test()

# Stage 2: 2B end-to-end (fast), prints per-language results as it goes
run_model("2b_base", MODEL_PATHS["2b_base"])
summarize()


Loading weights: 100%|██████████| 288/288 [00:00<00:00, 577.29it/s]


    dropping ' يعمل'/' يعملون': first tokens [157810]/[157810]
    dropping ' يلعب'/' يلعبون': first tokens [198386]/[198386]
    dropping ' يكتب'/' يكتبون': first tokens [4443]/[4443]
    note: ' anasoma'/' wanasoma' multi-token (3/3); scoring first token
    note: ' anacheza'/' wanacheza' multi-token (3/3); scoring first token
    note: ' anaandika'/' wanaandika' multi-token (3/3); scoring first token
clean=+2.8875 ablated=+2.2750 self-patched=+2.8750
self-test PASSED

===== 2b_base =====


Loading weights: 100%|██████████| 288/288 [00:00<00:00, 657.85it/s]


    dropping ' يعمل'/' يعملون': first tokens [157810]/[157810]
    dropping ' يلعب'/' يلعبون': first tokens [198386]/[198386]
    dropping ' يكتب'/' يكتبون': first tokens [4443]/[4443]
    note: ' anasoma'/' wanasoma' multi-token (3/3); scoring first token
    note: ' anacheza'/' wanacheza' multi-token (3/3); scoring first token
    note: ' anaandika'/' wanaandika' multi-token (3/3); scoring first token
  [fr] acc nat/matched 1.00/0.96 | retained EN-heads->lang +0.01 | compat +1.27 | floor +0.99
  [de] acc nat/matched 1.00/0.95 | retained EN-heads->lang -0.06 | compat +0.84 | floor +0.99
  [hi] acc nat/matched 0.80/0.94 | retained EN-heads->lang +0.87 | compat +3.26 | floor +0.99
  [sw] gate FAIL seed 0: matched acc 0.183 < 0.75
  [sw] gate FAIL seed 7: matched acc 0.175 < 0.75
  [sw] gate FAIL seed 13: matched acc 0.242 < 0.75
  [sw] acc nat/matched 0.01/0.20 | retained EN-heads->lang +nan | compat +nan | floor +nan
model    lang gate   acc_m  ret EN->L  ret L->L   compat   floor     

## Cell 10 - RUN (stage 2: 9B base, 9B it, 27B it)

In [10]:
# ===== RUN the rest =====  (only after the 2B output above looks sane)
for mk, path in MODEL_PATHS.items():
    if mk != "2b_base":
        run_model(mk, path)
summarize()



===== 9b_base =====


Loading weights: 100%|██████████| 464/464 [00:01<00:00, 273.73it/s]


    dropping ' يعمل'/' يعملون': first tokens [157810]/[157810]
    dropping ' يلعب'/' يلعبون': first tokens [198386]/[198386]
    dropping ' يكتب'/' يكتبون': first tokens [4443]/[4443]
    note: ' anasoma'/' wanasoma' multi-token (3/3); scoring first token
    note: ' anacheza'/' wanacheza' multi-token (3/3); scoring first token
    note: ' anaandika'/' wanaandika' multi-token (3/3); scoring first token
  ranking 9b_base/en: 672 heads x batched passes
    top-5 scores [0.47, 0.31, 0.22, 0.22, 0.21] | median 0.003
  ranking 9b_base/fr_m: 672 heads x batched passes
    top-5 scores [0.51, 0.36, 0.26, 0.25, 0.24] | median -0.003
  [fr] acc nat/matched 0.95/0.94 | retained EN-heads->lang +0.12 | compat +1.92 | floor +0.99
  ranking 9b_base/de_m: 672 heads x batched passes
    top-5 scores [0.35, 0.31, 0.25, 0.23, 0.22] | median 0.009
  [de] acc nat/matched 1.00/0.99 | retained EN-heads->lang +0.51 | compat +1.62 | floor +0.99
  ranking 9b_base/hi_m: 672 heads x batched passes
    top-5 sco

Loading weights: 100%|██████████| 464/464 [00:01<00:00, 251.68it/s]


    dropping ' يعمل'/' يعملون': first tokens [157810]/[157810]
    dropping ' يلعب'/' يلعبون': first tokens [198386]/[198386]
    dropping ' يكتب'/' يكتبون': first tokens [4443]/[4443]
    note: ' anasoma'/' wanasoma' multi-token (3/3); scoring first token
    note: ' anacheza'/' wanacheza' multi-token (3/3); scoring first token
    note: ' anaandika'/' wanaandika' multi-token (3/3); scoring first token
  ranking 9b_it/en: 672 heads x batched passes
    top-5 scores [0.75, 0.34, 0.28, 0.19, 0.17] | median 0.004
  ranking 9b_it/fr_m: 672 heads x batched passes
    top-5 scores [0.59, 0.29, 0.24, 0.21, 0.21] | median 0.008
  [fr] acc nat/matched 0.98/0.93 | retained EN-heads->lang +0.02 | compat +1.67 | floor +1.03
  ranking 9b_it/de_m: 672 heads x batched passes
    top-5 scores [0.42, 0.32, 0.15, 0.14, 0.14] | median -0.008
  [de] acc nat/matched 0.99/0.95 | retained EN-heads->lang +0.27 | compat +1.39 | floor +1.03
  ranking 9b_it/hi_m: 672 heads x batched passes
    top-5 scores [0.1

Loading weights: 100%|██████████| 508/508 [00:06<00:00, 83.16it/s] 


    dropping ' يعمل'/' يعملون': first tokens [157810]/[157810]
    dropping ' يلعب'/' يلعبون': first tokens [198386]/[198386]
    dropping ' يكتب'/' يكتبون': first tokens [4443]/[4443]
    note: ' anasoma'/' wanasoma' multi-token (3/3); scoring first token
    note: ' anacheza'/' wanacheza' multi-token (3/3); scoring first token
    note: ' anaandika'/' wanaandika' multi-token (3/3); scoring first token
  ranking 27b_it/en: 1472 heads x batched passes
    top-5 scores [0.58, 0.38, 0.24, 0.18, 0.15] | median 0.008
  ranking 27b_it/fr_m: 1472 heads x batched passes
    top-5 scores [0.31, 0.17, 0.17, 0.13, 0.13] | median 0.001
  [fr] acc nat/matched 0.94/0.80 | retained EN-heads->lang +0.77 | compat +4.32 | floor +1.02
  ranking 27b_it/de_m: 1472 heads x batched passes
    top-5 scores [0.21, 0.2, 0.16, 0.15, 0.13] | median -0.004
  [de] acc nat/matched 1.00/0.97 | retained EN-heads->lang +0.57 | compat +1.60 | floor +1.02
  ranking 27b_it/hi_m: 1472 heads x batched passes
    top-5 scor